In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone Code từ GitHub để đảm bảo lấy bản mới nhất (config.yaml, make_tsv.py)
%cd /content
!rm -rf Forget-MI-LoKU
!git clone https://github.com/nhnhu146/Forget-MI-LoKU.git
%cd Forget-MI-LoKU

# 3. Cài đặt các thư viện cần thiết
!pip install -q pydicom scikit-image wandb pyyaml pandas
!pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"

print("✅ Môi trường và mã nguồn đã sẵn sàng!")

In [ ]:
# 4. Giải nén Dữ liệu & Models từ Google Drive
# Lưu ý: Đảm bảo bạn đã upload 3 file zip vào thư mục MyDrive/Forget-MI-Project/
DRIVE_PATH = "/content/drive/MyDrive/Forget-MI-Project"

print("📦 Đang giải nén dữ liệu...")
# Giải nén Data (chứa img_data, metadata, text_data)
!unzip -q -o {DRIVE_PATH}/data.zip -d ./

# Giải nén Base Model vào folder forgetme/
!mkdir -p forgetme
!unzip -q -o {DRIVE_PATH}/base_model.zip -d ./forgetme/

# Giải nén Retrained Model (để so sánh)
!unzip -q -o {DRIVE_PATH}/retrained_model.zip -d ./

print("✅ Đã tải và giải nén toàn bộ Dữ liệu & Model!")

In [ ]:
# 5. Tiền xử lý dữ liệu (Tạo all_data.tsv)
!python make_tsv.py

# 6. Thiết lập thư mục lưu kết quả bền vững trên Drive
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
import os
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Tạo liên kết Symlink
if os.path.exists("unlearning_output"):
    !rm -rf unlearning_output
!ln -s {DRIVE_RESULTS} ./unlearning_output

print(f"✅ Đã kết nối thư mục Output với Drive: {DRIVE_RESULTS}")

In [ ]:
# 7. Chạy Unlearning LoKU
# Nhớ login wandb trước khi chạy nếu muốn log online: !wandb login
!WANDB_MODE=online python training/forgetmi_loku.py --config config.yaml